<a href="https://colab.research.google.com/github/ajaykumar080286/GENAI_LangChain/blob/main/20_tool_calling_in_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
!pip install -q langchain-openai langchain-core requests
import os
os.environ["OPENAI_API_KEY"]=""

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [4]:
@tool
def multiply(a :int , b: int) -> int:
  """Multiply two number and return result"""
  return a*b

In [5]:
multiply.invoke({"a":3,"b":4})

12

In [6]:
multiply.name

'multiply'

In [7]:
multiply.description

'Multiply two number and return result'

In [8]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [9]:
#tool binding
llm=ChatOpenAI(model="gpt-4o")

In [10]:
#llm.invoke("what is capital of india").content

In [11]:
#tools bining
llm_with_tool=llm.bind_tools([multiply])

In [12]:
query= HumanMessage("Can you multipy 3 with 1000")

In [13]:
message=[query]
message

[HumanMessage(content='Can you multipy 3 with 1000', additional_kwargs={}, response_metadata={})]

In [14]:
result=llm_with_tool.invoke(message)

In [15]:
message.append(result)

In [16]:
message

[HumanMessage(content='Can you multipy 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 57, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_be90e528c0', 'id': 'chatcmpl-Ek2yrEgu3W8fjXCqBDn68taGoaCJE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1d912-d1b5-76d6-a7d7-0967021f1b1c-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_tDumk1GPhIsr3OUBwtAnrbha', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 18, 'total_tokens': 75, 'in

In [17]:
#ool execution
total_result=multiply.invoke(result.tool_calls[0])

In [18]:
total_result

ToolMessage(content='3000', name='multiply', tool_call_id='call_tDumk1GPhIsr3OUBwtAnrbha')

In [19]:
message.append(total_result)

In [20]:
message

[HumanMessage(content='Can you multipy 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 57, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_be90e528c0', 'id': 'chatcmpl-Ek2yrEgu3W8fjXCqBDn68taGoaCJE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1d912-d1b5-76d6-a7d7-0967021f1b1c-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_tDumk1GPhIsr3OUBwtAnrbha', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 18, 'total_tokens': 75, 'in

In [21]:
llm_with_tool.invoke(message).content

'The result of multiplying 3 by 1000 is 3000.'

In [339]:
@tool
def get_conversion_factor(base_currency:str,target_currency: str )-> float :
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url=f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response=requests.get(url)
  return response.json()

@tool
def convert(base_currency_value:int,conversion_rate: float )-> float :

  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value*conversion_rate



In [340]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1794614401,
 'time_last_update_utc': 'Sat, 14 Nov 2026 00:00:01 +0000',
 'time_next_update_unix': 1794700801,
 'time_next_update_utc': 'Sun, 15 Nov 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.2897}

In [341]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [342]:
llm=ChatOpenAI(model="gpt-4o")

In [343]:
llm_with_tool=llm.bind_tools([get_conversion_factor,convert])

In [344]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 20 USD to INR')]

In [345]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 20 USD to INR', additional_kwargs={}, response_metadata={})]

In [346]:
ai_message=llm_with_tool.invoke(messages)


In [347]:
ai_message

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 123, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_fcbe8d60cc', 'id': 'chatcmpl-Ek4SXUJ97ubKddgZhqhVGgnUt0VUX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1d969-8ffa-7414-b219-62418797029d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_Ej8Ew9dYGJay5DvwvR2aZV4j', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 123, 'output_tokens': 22, 'total_tokens': 145, 'input_token_details': {'audio': 0, 'cache_read': 0}, 

In [348]:
messages.append(ai_message)

In [349]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 20 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 123, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_fcbe8d60cc', 'id': 'chatcmpl-Ek4SXUJ97ubKddgZhqhVGgnUt0VUX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1d969-8ffa-7414-b219-62418797029d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_Ej8Ew9dYGJay5DvwvR2aZV4j', 'type': '

In [350]:
first_call=get_conversion_factor.invoke(ai_message.tool_calls[0])

In [351]:
first_call

ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1794614401, "time_last_update_utc": "Sat, 14 Nov 2026 00:00:01 +0000", "time_next_update_unix": 1794700801, "time_next_update_utc": "Sun, 15 Nov 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 94.2897}', name='get_conversion_factor', tool_call_id='call_Ej8Ew9dYGJay5DvwvR2aZV4j')

In [352]:
messages.append(first_call)

In [353]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 20 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 123, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_fcbe8d60cc', 'id': 'chatcmpl-Ek4SXUJ97ubKddgZhqhVGgnUt0VUX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1d969-8ffa-7414-b219-62418797029d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_Ej8Ew9dYGJay5DvwvR2aZV4j', 'type': '

In [354]:
second_call=llm_with_tool.invoke(messages)

In [355]:
second_call

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 294, 'total_tokens': 317, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_fcbe8d60cc', 'id': 'chatcmpl-Ek4SYg70KL9aKFQcT6UTNWD8N3VEp', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1d969-9624-70b3-8209-82fcdbabe1a8-0', tool_calls=[{'name': 'convert', 'args': {'base_currency_value': 20, 'conversion_rate': 94.2897}, 'id': 'call_XjDy36QxJRRmLOE8psyXLccF', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 294, 'output_tokens': 23, 'total_tokens': 317, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_t

In [356]:
second_call.tool_calls[0]

{'name': 'convert',
 'args': {'base_currency_value': 20, 'conversion_rate': 94.2897},
 'id': 'call_XjDy36QxJRRmLOE8psyXLccF',
 'type': 'tool_call'}

In [357]:
third=convert.invoke(second_call.tool_calls[0])

In [358]:
messages.append(third)

In [359]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 20 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 123, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_fcbe8d60cc', 'id': 'chatcmpl-Ek4SXUJ97ubKddgZhqhVGgnUt0VUX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1d969-8ffa-7414-b219-62418797029d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_Ej8Ew9dYGJay5DvwvR2aZV4j', 'type': '

In [360]:
# Construct a properly structured message history
correct_messages = [
    messages[0],  # HumanMessage
    messages[1],  # AIMessage (get_conversion_factor tool_call)
    messages[2],  # ToolMessage (get_conversion_factor result)
    second_call,  # AIMessage (convert tool_call) <-- This was missing!
    third         # ToolMessage (convert result)
]

# Invoke the LLM with the correct sequence
llm_with_tool.invoke(correct_messages).content

'The conversion factor between USD and INR is 94.2897. Based on this conversion rate, 20 USD is equal to approximately 1885.79 INR.'